# POI Data Transformation Pipeline

## Overview
This notebook transforms **three silver-layer datasets** (`poi`, `employer`, `industry`) from `edbi_teamg02.silver` into an enriched, analysis-ready **gold layer** in `edbi_teamg02.gold`. The pipeline follows the medallion architecture established in the [Data Ingestion](/#notebook/1915875470418234) (bronze) and [POI EDA & QA](/#notebook/3304086644673254) (silver) notebooks.

## Data Sources
| Source Table | Records | Description |
| --- | --- | --- |
| `edbi_teamg02.silver.poi` | 7,242 | Person of Interest profiles (demographics, occupation, employer linkage) |
| `edbi_teamg02.silver.employer` | 2,700 | Employer registry (entity details, address, SSIC codes, officer counts) |
| `edbi_teamg02.silver.industry` | 988 | SSIC industry classification hierarchy (section → sub-class) |

## Quality Assurance
**Pre-transformation profiling** is performed before any data manipulation:
* **Null analysis** across all columns in all three tables, with percentage breakdowns
* **Primary key uniqueness** validated (NRIC, UEN, SSIC) — zero duplicates confirmed
* **Join key coverage** quantified: 100% of non-null POI UENs match an employer; 14 of 326 employer SSIC codes have no industry match
* **Column naming consistency** audited — `df_poi` standardised from Title Case with spaces to snake_case
* **Data gap analysis** on unlinked POIs (2,007 records, 54.9% unemployed) and unmatched SSIC codes (86 POI rows across 14 legacy codes)
* **Boolean flag columns** (`has_employer`, `has_industry`, `is_unemployed`) explicitly mark data gaps for downstream filtering

**Post-transformation validation** runs 8 automated checks before the gold save:
* Row count preservation (no fan-out from joins)
* Primary key uniqueness maintained after all transformations
* No unexpected nulls in core source columns
* Range checks on derived features (`age` 0–120, `company_age` ≥ 0)
* Derived features are populated wherever source data exists
* Flag column consistency (`has_employer` aligns with `entity_name`)
* Mapping completeness for `occupation_sector` and `nationality_region`
* Employer-only columns are null for unlinked POIs

## Data Integration
Three data sources are integrated via a two-stage left join chain:
1. **POI → Employer** on `uen_identifier` → `uen` (left join preserves 2,007 unlinked POIs)
2. **Result → Industry** on `primary_ssic_code` → `ssic` (left join preserves 86 rows with unmatched SSIC)

All data is consistently formatted and standardised: column names are snake_case, date columns are parsed to datetime, and numeric columns cast from float64 back to nullable Int64 after join-introduced NaNs.

## Transformation Techniques
The following derived features are engineered from the enriched dataset:

| Feature | Technique | Source Columns |
| --- | --- | --- |
| `age` | Date parsing + arithmetic (relative to `year_of_profile`) | `dob`, `year_of_profile` |
| `company_age` | Date parsing + arithmetic | `registration_incorporation_date`, `year_of_profile` |
| `age_group` | Binning into 7 demographic brackets (\<18 to 65+) | `age` |
| `company_age_group` | Binning into 6 lifecycle stages (Startup to Heritage) | `company_age` |
| `poi_per_employer` | Group-level aggregation (transform) | `uen_identifier`, `nric` |
| `postal_district` | Postal code mapping to 28 Singapore districts (D01–D28) | `postal_code` |
| `full_address` | Vectorised multi-field concatenation into SG address format | `block`, `street_name`, `level_no`, `unit_no`, `building_name`, `postal_code` |
| `nationality_region` | Mapping \~200 nationalities into 13 broad regions | `nationality` |
| `occupation_sector` | Mapping 41 SSOC titles into 9 sectors (ISCO major groups) | `occupation` |
| `occupation_skill_level` | Ordinal 1–4 skill tier (ISCO classification convention) | `occupation_sector` |

## Gold Layer Output
The pipeline produces a **star schema** with one fact table and three dimension tables, written as Delta tables with `delta.columnMapping.mode = name` and configurable partition keys:

| Gold Table | Records | Columns | Partition Key | Description |
| --- | --- | --- | --- | --- |
| `fact_poi_enriched` | 7,242 | 51 | `year_of_profile` | Fully denormalized analytical table |
| `dim_poi` | 7,242 | 19 | `year_of_profile` | POI attributes + derived person-level features |
| `dim_employer` | 2,286 | 22 | `entity_status_description` | Employer attributes + derived company-level features |
| `dim_industry` | 988 | 11 | `section_code` | Industry classification reference (from silver) |

## Efficiency
* **Vectorised pandas operations** used throughout — address concatenation uses vectorised Series construction before a lightweight join, avoiding row-by-row iteration over the full enriched DataFrame
* **Reusable helper functions** (`write_to_gold` with optional `partition_by`, `null_profile`) standardised with the bronze/silver layer patterns
* **Partitioned Delta tables** enable predicate pushdown for common query filters (year, entity status, industry section)
* **Automated end-to-end pipeline**: running all cells sequentially produces the complete gold schema from silver inputs with no manual intervention required

In [0]:
import sys
sys.path.insert(0, "/Workspace/Users/marcus_lim@cpib.gov.sg")

from poi_utils import enrich_poi, prepare_for_spark

# Read silver tables from catalog and convert to pandas
df_poi = spark.read.table("edbi_teamg02.silver.poi").toPandas()
df_employer = spark.read.table("edbi_teamg02.silver.employer").toPandas()
df_industry = spark.read.table("edbi_teamg02.silver.industry").toPandas()

print(f"df_poi: {df_poi.shape}")
print(f"df_employer: {df_employer.shape}")
print(f"df_industry: {df_industry.shape}")

df_poi: (7242, 12)
df_employer: (2700, 17)
df_industry: (988, 11)


In [0]:
import pandas as pd

# ── Helper: null summary ──
def null_profile(df, name):
    nulls = df.isnull().sum()
    pct = (nulls / len(df) * 100).round(2)
    summary = pd.DataFrame({"column": nulls.index, "null_count": nulls.values, "null_pct": pct.values, "dtype": df.dtypes.astype(str).values})
    summary = summary[summary["null_count"] > 0].sort_values("null_pct", ascending=False).reset_index(drop=True)
    return summary

print("=" * 70)
print("1. NULL ANALYSIS")
print("=" * 70)
for name, df in [("df_poi", df_poi), ("df_employer", df_employer), ("df_industry", df_industry)]:
    profile = null_profile(df, name)
    if profile.empty:
        print(f"\n[{name}] No nulls found.")
    else:
        print(f"\n[{name}] Columns with nulls ({len(profile)}/{df.shape[1]}):")
        print(profile.to_string(index=False))

print("\n" + "=" * 70)
print("2. DUPLICATE ANALYSIS (on primary keys)")
print("=" * 70)
poi_nric_dupes = df_poi["NRIC"].duplicated().sum()
emp_uen_dupes = df_employer["uen"].duplicated().sum()
ind_ssic_dupes = df_industry["ssic"].duplicated().sum()
print(f"  df_poi   — duplicate NRICs:       {poi_nric_dupes} / {len(df_poi)}")
print(f"  df_employer — duplicate UENs:      {emp_uen_dupes} / {len(df_employer)}")
print(f"  df_industry — duplicate SSICs:     {ind_ssic_dupes} / {len(df_industry)}")

print("\n" + "=" * 70)
print("3. JOIN KEY COVERAGE")
print("=" * 70)

# POI → Employer (UEN Identifier → uen)
poi_uens = df_poi["UEN Identifier"].dropna().unique()
emp_uens = set(df_employer["uen"].dropna().unique())
poi_matched = sum(1 for u in poi_uens if u in emp_uens)
poi_uen_null = df_poi["UEN Identifier"].isnull().sum()
print(f"\n  POI → Employer (UEN Identifier → uen):")
print(f"    POI rows with null UEN Identifier: {poi_uen_null} / {len(df_poi)} ({poi_uen_null/len(df_poi)*100:.1f}%)")
print(f"    Distinct POI UENs matched in employer: {poi_matched} / {len(poi_uens)}")
print(f"    Unmatched POI UENs: {len(poi_uens) - poi_matched}")

# Employer → Industry (primary_ssic_code → ssic)
emp_ssic = df_employer["primary_ssic_code"].dropna().unique()
ind_ssic = set(df_industry["ssic"].dropna().unique())
emp_matched = sum(1 for s in emp_ssic if s in ind_ssic)
emp_ssic_null = df_employer["primary_ssic_code"].isnull().sum()
print(f"\n  Employer → Industry (primary_ssic_code → ssic):")
print(f"    Employer rows with null primary_ssic_code: {emp_ssic_null} / {len(df_employer)} ({emp_ssic_null/len(df_employer)*100:.1f}%)")
print(f"    Distinct SSIC codes matched in industry: {emp_matched} / {len(emp_ssic)}")
print(f"    Unmatched SSIC codes: {len(emp_ssic) - emp_matched}")

print("\n" + "=" * 70)
print("4. COLUMN NAME CONSISTENCY")
print("=" * 70)
for name, df in [("df_poi", df_poi), ("df_employer", df_employer), ("df_industry", df_industry)]:
    cols_with_spaces = [c for c in df.columns if " " in c]
    cols_with_upper = [c for c in df.columns if c != c.lower()]
    if cols_with_spaces or cols_with_upper:
        print(f"  [{name}] Needs standardisation:")
        if cols_with_spaces:
            print(f"    Columns with spaces: {cols_with_spaces}")
        if cols_with_upper:
            print(f"    Columns with uppercase: {cols_with_upper}")
    else:
        print(f"  [{name}] Column names are clean.")

1. NULL ANALYSIS

[df_poi] Columns with nulls (1/12):
        column  null_count  null_pct  dtype
UEN Identifier        2007     27.71 object

[df_employer] Columns with nulls (8/17):
                           column  null_count  null_pct  dtype
                    building_name        1781     65.96 object
              secondary_ssic_code        1644     60.89 object
                          unit_no        1582     58.59 object
                         level_no        1571     58.19 object
business_constitution_description         296     10.96 object
                            block          29      1.07 object
                      street_name          11      0.41 object
                      postal_code           3      0.11 object

[df_industry] No nulls found.

2. DUPLICATE ANALYSIS (on primary keys)
  df_poi   — duplicate NRICs:       0 / 7242
  df_employer — duplicate UENs:      0 / 2700
  df_industry — duplicate SSICs:     0 / 988

3. JOIN KEY COVERAGE

  POI → Employer (

In [0]:
# All transformations handled by enrich_poi() from poi_utils:
# - Standardise POI column names (Title Case → snake_case)
# - Left join POI → Employer (on uen_identifier = uen)
# - Left join Result → Industry (on primary_ssic_code = ssic)
# - Add flags: has_employer, has_industry, is_unemployed
# - Feature engineering: age, company_age, poi_per_employer, bins,
#   postal_district, full_address, nationality_region, occupation_sector/skill
df_enriched = enrich_poi(df_poi, df_employer, df_industry)

print(f"Enriched table: {df_enriched.shape[0]} rows × {df_enriched.shape[1]} columns")
print(f"\nColumns:\n{list(df_enriched.columns)}")

  Enriched: 7242 rows, 53 cols
  Employer matched: 5235 / 7242
  Industry matched: 5149 / 7242
Enriched table: 7242 rows × 53 columns

Columns:
['nric', 'name', 'race', 'gender', 'email', 'marital_status', 'nationality', 'dob', 'phone_number', 'occupation', 'uen_identifier', 'year_of_profile', 'building_name', 'secondary_ssic_code', 'unit_no', 'level_no', 'business_constitution_description', 'block', 'street_name', 'postal_code', 'registration_incorporation_date', 'entity_name', 'entity_type_description', 'id', 'entity_status_description', 'uen_issue_date', 'primary_ssic_code', 'no_of_officers', 'section_code', 'division_code', 'group_code', 'class_code', 'sub_class_code', 'section', 'division', 'group', 'class', 'sub_class', 'has_employer', 'has_industry', 'is_unemployed', 'dob_parsed', 'reg_date_parsed', 'age', 'company_age', 'poi_per_employer', 'age_group', 'company_age_group', 'postal_district', 'full_address', 'nationality_region', 'occupation_sector', 'occupation_skill_level']


In [0]:
# ══════════════════════════════════════════════════════════════════════
# 1. UNLINKED POIs — no UEN identifier (cannot join to any employer)
# ══════════════════════════════════════════════════════════════════════
df_no_uen = df_enriched[df_enriched["uen_identifier"].isna()]

print("=" * 70)
print("1. POIs WITHOUT UEN IDENTIFIER (unlinked to any employer)")
print("=" * 70)
print(f"  Count: {len(df_no_uen)} / {len(df_enriched)} ({len(df_no_uen)/len(df_enriched)*100:.1f}%)")
print(f"\n  Demographic breakdown:")
print(f"    Gender:")
for val, cnt in df_no_uen["gender"].value_counts().items():
    print(f"      {val}: {cnt} ({cnt/len(df_no_uen)*100:.1f}%)")
print(f"    Nationality:")
for val, cnt in df_no_uen["nationality"].value_counts().head(5).items():
    print(f"      {val}: {cnt} ({cnt/len(df_no_uen)*100:.1f}%)")
print(f"    Top occupations:")
for val, cnt in df_no_uen["occupation"].value_counts().head(5).items():
    print(f"      {val}: {cnt} ({cnt/len(df_no_uen)*100:.1f}%)")

# ══════════════════════════════════════════════════════════════════════
# 2. UNMATCHED SSIC CODES — employer SSIC not found in industry table
# ══════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("2. EMPLOYERS WITH UNMATCHED SSIC CODES (no industry classification)")
print("=" * 70)

# Rows that have employer data but no industry match
df_no_industry = df_enriched[
    df_enriched["entity_name"].notna() & df_enriched["section"].isna()
]

print(f"  POI rows affected: {len(df_no_industry)} / {df_enriched['entity_name'].notna().sum()} linked POIs")

# Get distinct unmatched SSIC codes with their employer counts
unmatched_ssic = (
    df_no_industry
    .groupby("primary_ssic_code")
    .agg(employer_count=("entity_name", "nunique"), poi_count=("nric", "count"))
    .reset_index()
    .sort_values("poi_count", ascending=False)
)
print(f"  Distinct unmatched SSIC codes: {len(unmatched_ssic)}")
print(f"\n  Unmatched SSIC code detail:")
for _, row in unmatched_ssic.iterrows():
    print(f"    SSIC {int(row['primary_ssic_code'])}: {int(row['employer_count'])} employer(s), {int(row['poi_count'])} POI(s)")

# Show sample employers with unmatched SSIC
print(f"\n  Sample employers with unmatched SSIC:")
sample_cols = ["entity_name", "primary_ssic_code", "entity_status_description", "entity_type_description"]
print(df_no_industry[sample_cols].drop_duplicates().head(10).to_string(index=False))

1. POIs WITHOUT UEN IDENTIFIER (unlinked to any employer)
  Count: 2007 / 7242 (27.7%)

  Demographic breakdown:
    Gender:
      M: 1031 (51.4%)
      F: 976 (48.6%)
    Nationality:
      N.A.: 299 (14.9%)
      Reunionese: 17 (0.8%)
      Liechtensteiner: 15 (0.7%)
      Surinamer: 15 (0.7%)
      Equatorial Guinea: 15 (0.7%)
    Top occupations:
      Unemployed: 1101 (54.9%)
      Business & Administration Associate Professionals: 97 (4.8%)
      Science & Engineering Professionals: 87 (4.3%)
      Administrative & Commercial Managers: 82 (4.1%)
      Business & Administration Professionals: 68 (3.4%)

2. EMPLOYERS WITH UNMATCHED SSIC CODES (no industry classification)
  POI rows affected: 86 / 5235 linked POIs
  Distinct unmatched SSIC codes: 14

  Unmatched SSIC code detail:
    SSIC 47420: 12 employer(s), 21 POI(s)
    SSIC 10613: 5 employer(s), 14 POI(s)
    SSIC 85408: 4 employer(s), 10 POI(s)
    SSIC 85409: 3 employer(s), 10 POI(s)
    SSIC 10765: 3 employer(s), 6 POI(s)
 

In [0]:
# ══════════════════════════════════════════════════════════════════════
# Post-transformation validation — verify data integrity before gold save
# ══════════════════════════════════════════════════════════════════════

EXPECTED_ROWS = 7242  # POI source count after silver QA
validation_pass = True
errors = []

# 1. Row count preserved (no fan-out from joins)
if len(df_enriched) != EXPECTED_ROWS:
    errors.append(f"Row count mismatch: expected {EXPECTED_ROWS}, got {len(df_enriched)}")

# 2. Primary key uniqueness maintained
nric_dupes = df_enriched["nric"].duplicated().sum()
if nric_dupes > 0:
    errors.append(f"NRIC duplicates introduced: {nric_dupes}")

# 3. Core source columns have no unexpected nulls
for col in ["nric", "name", "race", "gender", "nationality", "dob", "occupation"]:
    null_count = df_enriched[col].isna().sum()
    if null_count > 0:
        errors.append(f"Unexpected nulls in source column '{col}': {null_count}")

# 4. Derived feature range checks
if df_enriched["age"].min() < 0:
    errors.append(f"Negative age detected: min={df_enriched['age'].min()}")
if df_enriched["age"].max() > 120:
    errors.append(f"Unreasonable age detected: max={df_enriched['age'].max()}")
if df_enriched["company_age"].dropna().min() < 0:
    errors.append(f"Negative company_age detected: min={df_enriched['company_age'].dropna().min()}")

# 5. Derived features have no nulls where source data exists
age_null_but_dob_exists = (df_enriched["age"].isna() & df_enriched["dob"].notna()).sum()
if age_null_but_dob_exists > 0:
    errors.append(f"Age is null despite DOB existing: {age_null_but_dob_exists} rows")

company_age_gap = (df_enriched["company_age"].isna() & df_enriched["has_employer"]).sum()
if company_age_gap > 0:
    errors.append(f"Company age null despite employer match: {company_age_gap} rows")

# 6. Flag column consistency
flag_mismatch = (df_enriched["has_employer"] & df_enriched["entity_name"].isna()).sum()
if flag_mismatch > 0:
    errors.append(f"has_employer=True but entity_name is null: {flag_mismatch} rows")

# 7. Mapping completeness
for col, label in [("occupation_sector", "Occupation sector"), ("nationality_region", "Nationality region")]:
    unknown_count = (df_enriched[col] == "Unknown").sum()
    if unknown_count > 0:
        unmapped = df_enriched.loc[df_enriched[col] == "Unknown", col.replace("_sector", "").replace("_region", "")].unique()
        errors.append(f"{label} has {unknown_count} 'Unknown' mappings from: {list(unmapped)[:5]}")

# 8. Employer-only columns are null for unlinked POIs
emp_cols_spot = ["entity_name", "postal_district", "company_age"]
for col in emp_cols_spot:
    leaked = (df_enriched[col].notna() & ~df_enriched["has_employer"]).sum()
    if leaked > 0:
        errors.append(f"'{col}' has values for unlinked POIs: {leaked} rows")

# RESULTS
print("=" * 70)
print("POST-TRANSFORMATION VALIDATION")
print("=" * 70)
print(f"  Enriched table: {df_enriched.shape[0]} rows, {df_enriched.shape[1]} cols")
print(f"  Expected rows:  {EXPECTED_ROWS}")
print()
if errors:
    print(f"  ⚠️  {len(errors)} ISSUE(S) FOUND:")
    for i, err in enumerate(errors, 1):
        print(f"    {i}. {err}")
else:
    print("  ✅ ALL CHECKS PASSED — data is ready for gold layer save")

POST-TRANSFORMATION VALIDATION
  Enriched table: 7242 rows, 53 cols
  Expected rows:  7242

  ⚠️  1 ISSUE(S) FOUND:
    1. Nationality region has 1204 'Unknown' mappings from: ['N.A.', 'Bahamian', 'Unknown', 'Others', 'Nationality']


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS edbi_teamg02.gold;

In [0]:
# ══════════════════════════════════════════════════════════════════════
# Save enriched data to gold schema as Delta tables
# Uses prepare_for_spark() from poi_utils for type conversion
# ══════════════════════════════════════════════════════════════════════

def write_to_gold(df, table_name, partition_by=None):
    """Write a pandas DataFrame to the gold layer as a Delta table."""
    df = prepare_for_spark(df)
    spark_df = spark.createDataFrame(df)
    writer = spark_df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .option("delta.columnMapping.mode", "name")
    if partition_by:
        cols = [partition_by] if isinstance(partition_by, str) else partition_by
        writer = writer.partitionBy(*cols)
    writer.saveAsTable(f"edbi_teamg02.gold.{table_name}")
    part_msg = f" (partitioned by {partition_by})" if partition_by else ""
    print(f"\n edbi_teamg02.gold.{table_name} written — {spark_df.count()} rows{part_msg}")

# ── 1. FACT TABLE: partitioned by year_of_profile (10 values, natural time filter) ──
fact_cols = [c for c in df_enriched.columns if c not in ["dob_parsed", "reg_date_parsed"]]
df_fact = df_enriched[fact_cols].copy()
write_to_gold(df_fact, "fact_poi_enriched", partition_by="year_of_profile")

# ── 2. DIM_POI: partitioned by year_of_profile (aligns with fact table) ──
poi_dim_cols = [
    "nric", "name", "race", "gender", "email", "marital_status",
    "nationality", "nationality_region", "dob", "phone_number",
    "occupation", "occupation_sector", "occupation_skill_level",
    "year_of_profile", "age", "age_group",
    "uen_identifier", "has_employer", "is_unemployed"
]
df_dim_poi = df_enriched[poi_dim_cols].drop_duplicates(subset=["nric"]).copy()
write_to_gold(df_dim_poi, "dim_poi", partition_by="year_of_profile")

# ── 3. DIM_EMPLOYER: partitioned by entity_status_description (common filter) ──
emp_dim_cols = [
    "uen_identifier", "entity_name", "entity_type_description",
    "business_constitution_description", "entity_status_description",
    "registration_incorporation_date", "uen_issue_date",
    "primary_ssic_code", "secondary_ssic_code", "no_of_officers",
    "block", "street_name", "level_no", "unit_no", "building_name",
    "postal_code", "postal_district", "full_address",
    "company_age", "company_age_group", "poi_per_employer",
    "has_industry"
]
df_dim_emp = (
    df_enriched[df_enriched["has_employer"]][emp_dim_cols]
    .drop_duplicates(subset=["uen_identifier"])
    .copy()
)
write_to_gold(df_dim_emp, "dim_employer", partition_by="entity_status_description")

# ── 4. DIM_INDUSTRY: partitioned by section_code (~21 top-level SSIC sections) ──
write_to_gold(df_industry.copy(), "dim_industry", partition_by="section_code")


 edbi_teamg02.gold.fact_poi_enriched written — 7242 rows (partitioned by year_of_profile)

 edbi_teamg02.gold.dim_poi written — 7242 rows (partitioned by year_of_profile)

 edbi_teamg02.gold.dim_employer written — 2286 rows (partitioned by entity_status_description)

 edbi_teamg02.gold.dim_industry written — 988 rows (partitioned by section_code)
